# Combined on+off arb, single listing, `highAtStartGoLow` marker -- scope test

Purpose: test a DIFFERENT way to build CH2's sequence that -- if it works
as hoped -- would eliminate the ~250-listing sequence-table ceiling found
earlier (see `notes.md`'s "spurious off-resonance/no-MW-near-sample
signal" entry) while keeping the EXACT SAME slow block-chop timing
(`n_reps` reps of "on", then `n_reps` reps of "off", repeating).

Current `rabi.py` design: TWO listings per off+on cycle --
`["gate_off_rep", n_reps, repeat, lowAtStart]`,
`["gate_on_rep"/on_segment_arb, n_reps, repeat, highAtStart]` -- each
relying on the segment's own cheap `repeat_count` field to replay a
SHORT (one-rep-long) arb `n_reps` times. Repeating this pair
`anchor_free_reps` times costs `2 * anchor_free_reps` table entries,
which is what hit the ~250-listing ceiling.

This test instead pre-bakes `n_reps` copies of "on" content followed by
`n_reps` copies of "off" content into ONE combined arb (per the Keysight
manual's `highAtStartGoLow` marker mode: "force marker high at start of
segment and then low at marker position", where `<marker point>` is a
sample index into the arb, `4 <= marker_point <= N-3`), then lists that
ONE arb with `repeat_count = anchor_free_reps` -- a SINGLE table entry
regardless of how large `anchor_free_reps` is, same as how `n_reps`
itself costs nothing extra today.

**The open question this notebook tests**: does `highAtStartGoLow`'s
assert-then-negate pattern re-fire on EVERY repeat of the arb (what we
need -- each of the `anchor_free_reps` repeats should independently show
HIGH-then-LOW), or does it only fire ONCE across the whole multi-repeat
block (which would break this approach)? The manual doesn't say
explicitly, and the only previously-confirmed real-hardware finding (see
notes.md) was for the SIMPLER `highAtStart` mode (no negate), which
plausibly behaves differently since it has nothing to "redo" each repeat.
This notebook does NOT modify `rabi.py` -- it's a standalone scope test.

Small `N_REPS`/`ANCHOR_FREE_REPS` here so individual repeats and their
marker transitions are easy to count and verify on a scope.

## Connect to the AWG + SDG1062X

In [1]:
import sys
sys.path.insert(0, "..")

import time
import numpy as np

import rabi
import ks33600a
import sdg1062x

awg = ks33600a.KS33600A(rabi.AWG_RESOURCE, debug=True)
sdg = sdg1062x.SDG1062X(rabi.SDG_RESOURCE, debug=True)
print("Connected to AWG and SDG1062X.")


Keysight 33600A: connected
*RST => +0,"No error"
*CLS => +0,"No error"
SOUR1:DATA:VOL:CLE => +0,"No error"
SOUR2:DATA:VOL:CLE => +0,"No error"
*RST
Siglent SDG1062X: connected
Connected to AWG and SDG1062X.


## Parameters

`N_REPS` (per on/off phase) and `ANCHOR_FREE_REPS` (how many times the
ONE combined arb repeats before needing a fresh trigger) are both kept
small so you can visually count individual repeats and confirm the
marker re-asserts HIGH at the start of EACH one, not just the first.

In [7]:
LASER_US = 2.0
PRE_US = 1.0
MW_US = 2.0
POST_US = 1.0
N_REPS = 5              # small -- real sweeps use 250
ANCHOR_FREE_REPS = 1000    # repeats of the ONE combined arb -- small so you
                        # can count them and check EACH one shows the
                        # marker transition, not just the first

CH1_VPP = 0.632
CH2_VPP = 5.0
CH2_OFFSET_V = 2.5
SAMPLE_RATE_HZ = 1e9

SEQUENCE_NAME_CH1 = "combined_test_ch1"
SEQUENCE_NAME_CH2 = "combined_test_ch2"


## Build the combined arbs

CH2: `[gate_on_rep x N_REPS] + [gate_off_rep x N_REPS]` as ONE array,
`marker_point` = sample index where the off-portion begins (i.e. the
length of the on-portion). CH1: `n_reps*2` copies of the same "rep" arb
concatenated (CH1 never distinguishes on/off, so no marker transition
needed -- `maintain` mode is enough), also collapsed to one listing.

In [8]:
laser_samples = rabi._us_to_samples(LASER_US, SAMPLE_RATE_HZ)
pre_samples = rabi._us_to_samples(PRE_US, SAMPLE_RATE_HZ)
mw_samples = rabi._us_to_samples(MW_US, SAMPLE_RATE_HZ)
post_samples = rabi._us_to_samples(POST_US, SAMPLE_RATE_HZ)

ch1_rep = np.concatenate([
    rabi._rf_pulse(80e6, laser_samples, SAMPLE_RATE_HZ),
    rabi._const(pre_samples + mw_samples + post_samples, 0.0),
])
gate_on_rep = np.concatenate([
    rabi._const(laser_samples + pre_samples, -1.0),
    rabi._const(mw_samples, 1.0),
    rabi._const(post_samples, -1.0),
])
gate_off_rep = rabi._const(laser_samples + pre_samples + mw_samples + post_samples, -1.0)

assert len(ch1_rep) == len(gate_on_rep) == len(gate_off_rep)

# CH2: one combined arb, on-content first then off-content, N_REPS copies each.
ch2_combined = np.concatenate([gate_on_rep] * N_REPS + [gate_off_rep] * N_REPS)
marker_point = len(gate_on_rep) * N_REPS
print(f"ch2_combined: {len(ch2_combined)} samples, marker_point={marker_point} "
      f"(must be in [4, {len(ch2_combined) - 3}])")
assert 4 <= marker_point <= len(ch2_combined) - 3

# CH1: one combined arb, n_reps*2 copies of the same "rep" (no marker transition needed).
ch1_combined = np.concatenate([ch1_rep] * (2 * N_REPS))
print(f"ch1_combined: {len(ch1_combined)} samples")

anchor_ch1 = rabi._const(rabi.ANCHOR_SAMPLES, 0.0)
anchor_ch2 = rabi._const(rabi.ANCHOR_SAMPLES, -1.0)

awg.write("SOUR1:DATA:VOL:CLE")
awg.write("SOUR2:DATA:VOL:CLE")
awg.upload_waveform(ch1_combined, arb_name="ch1_combined", ch=1, sample_rate=SAMPLE_RATE_HZ)
awg.upload_waveform(anchor_ch1, arb_name="anchor", ch=1, sample_rate=SAMPLE_RATE_HZ)
awg.upload_waveform(ch2_combined, arb_name="ch2_combined", ch=2, sample_rate=SAMPLE_RATE_HZ)
awg.upload_waveform(anchor_ch2, arb_name="anchor", ch=2, sample_rate=SAMPLE_RATE_HZ)
print("Arbs uploaded.")


ch2_combined: 60000 samples, marker_point=30000 (must be in [4, 59997])
ch1_combined: 60000 samples
SOUR1:DATA:VOL:CLE => +0,"No error"
SOUR2:DATA:VOL:CLE => +0,"No error"
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 1
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 1
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 2
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 2
Arbs uploaded.


## Build the sequences: ONE listing per channel (plus anchor)

CH2 uses `highAtStartGoLow` with the computed `marker_point` -- this is
the part being tested. CH1 uses plain `maintain`.

In [9]:
block1 = rabi.build_block_descriptor(SEQUENCE_NAME_CH1, [
    ["anchor", "1", "onceWaitTrig", "maintain", 10],
    ["ch1_combined", str(ANCHOR_FREE_REPS), "repeat", "maintain", 10],
])
awg.write(f"DATA:SEQ {block1}")

block2 = rabi.build_block_descriptor(SEQUENCE_NAME_CH2, [
    ["anchor", "1", "onceWaitTrig", "lowAtStart", 10],
    ["ch2_combined", str(ANCHOR_FREE_REPS), "repeat", "highAtStartGoLow", str(marker_point)],
])
awg.write(f"SOUR2:DATA:SEQ {block2}")
print("Sequences written.")


DATA:SEQ #294"combined_test_ch1","anchor",1,onceWaitTrig,maintain,10,"ch1_combined",1000,repeat,maintain,10 => +0,"No error"
SOUR2:DATA:SEQ #3107"combined_test_ch2","anchor",1,onceWaitTrig,lowAtStart,10,"ch2_combined",1000,repeat,highAtStartGoLow,30000 => +0,"No error"
Sequences written.


## Configure external trigger + channel outputs

Trigger set to a literal, slow, fixed rate so individual edges are
visible on the scope, same approach as the earlier anchor_free_reps
test.

In [10]:
TRIGGER_HZ = 10.0
sdg.write("C1:BSWV WVTP,SQUARE")
sdg.write(f"C1:BSWV FRQ,{TRIGGER_HZ}")
sdg.write("C1:BSWV AMP,5")
sdg.write("C1:BSWV OFST,2.5")
sdg.write("C1:OUTP ON")

# CH1
awg.write("OUTP1:LOAD 50")
awg.write(f"SOUR1:FUNC:ARB:SRAT {SAMPLE_RATE_HZ}")
awg.write(f'SOUR1:FUNC:ARB "{SEQUENCE_NAME_CH1}"')
awg.write("SOUR1:FUNC ARB")
awg.write(f"SOUR1:VOLT {CH1_VPP}")
awg.write("TRIG1:SOUR EXT")
awg.write("TRIG1:SLOP POS")
awg.write("TRIG1:LEV 1.5")
awg.write("OUTPUT1 ON")

# CH2
awg.write("OUTP2:LOAD INF")
awg.write(f"SOUR2:FUNC:ARB:SRAT {SAMPLE_RATE_HZ}")
awg.write(f'SOUR2:FUNC:ARB "{SEQUENCE_NAME_CH2}"')
awg.write("SOUR2:FUNC ARB")
awg.write(f"SOUR2:VOLT {CH2_VPP}")
awg.write(f"SOUR2:VOLT:OFFS {CH2_OFFSET_V}")
awg.write("TRIG2:SOUR EXT")
awg.write("TRIG2:SLOP POS")
awg.write("TRIG2:LEV 1.5")
awg.write("OUTPUT2 ON")

awg.write("OUTPut:SYNC:SOURce CH2")
print(f"Trigger fixed at {TRIGGER_HZ} Hz. Outputs on, Sync routed to CH2.")


C1:BSWV WVTP,SQUARE
C1:BSWV FRQ,10.0
C1:BSWV AMP,5
C1:BSWV OFST,2.5
C1:OUTP ON
OUTP1:LOAD 50 => +0,"No error"
SOUR1:FUNC:ARB:SRAT 1000000000.0 => +0,"No error"
SOUR1:FUNC:ARB "combined_test_ch1" => +0,"No error"
SOUR1:FUNC ARB => +0,"No error"
SOUR1:VOLT 0.632 => +0,"No error"
TRIG1:SOUR EXT => +0,"No error"
TRIG1:SLOP POS => +0,"No error"
TRIG1:LEV 1.5 => +0,"No error"
OUTPUT1 ON => +0,"No error"
OUTP2:LOAD INF => +0,"No error"
SOUR2:FUNC:ARB:SRAT 1000000000.0 => +0,"No error"
SOUR2:FUNC:ARB "combined_test_ch2" => +0,"No error"
SOUR2:FUNC ARB => +0,"No error"
SOUR2:VOLT 5.0 => +0,"No error"
SOUR2:VOLT:OFFS 2.5 => +0,"No error"
TRIG2:SOUR EXT => +0,"No error"
TRIG2:SLOP POS => +0,"No error"
TRIG2:LEV 1.5 => +0,"No error"
OUTPUT2 ON => +0,"No error"
OUTPut:SYNC:SOURce CH2 => +0,"No error"
Trigger fixed at 10.0 Hz. Outputs on, Sync routed to CH2.


## Diagnostic: confirm the instrument's actual state

In [ ]:
print("OUTP1?", awg.query("OUTP1?"))
print("OUTP2?", awg.query("OUTP2?"))
print("SOUR1:FUNC:ARB?", awg.query("SOUR1:FUNC:ARB?"))
print("SOUR2:FUNC:ARB?", awg.query("SOUR2:FUNC:ARB?"))
print("SOUR1:VOLT?", awg.query("SOUR1:VOLT?"))
print("SOUR2:VOLT?", awg.query("SOUR2:VOLT?"))
print("OUTPut:SYNC:SOURce?", awg.query("OUTPut:SYNC:SOURce?"))
print("SYST:ERR?", awg.query("SYST:ERR?"))


## What to check on the oscilloscope

- **First: confirm the `highAtStartGoLow` SCPI syntax was even accepted**
  -- check `SYST:ERR?` above came back clean (no `-2xx` parse/parameter
  error). If it errored, the literal name or `marker_point` units/range
  are wrong and need revisiting before anything else matters.
- **Sync/Marker BNC output (the critical test)**: should show `N_REPS`
  reps' worth of HIGH, then `N_REPS` reps' worth of LOW, repeating this
  pattern `ANCHOR_FREE_REPS` times total before the brief anchor-wait
  pause. Count the HIGH/LOW blocks carefully -- if the marker only goes
  HIGH once (at the very first repeat) and then stays LOW for the
  remaining `ANCHOR_FREE_REPS - 1` repeats, `highAtStartGoLow` does NOT
  retrigger per-repeat and this whole approach doesn't work as hoped. If
  it correctly shows the HIGH/LOW pattern repeating `ANCHOR_FREE_REPS`
  times, it works.
- **CH2 analog output**: should show real gating (low during the on-block
  content... note CH2's OWN "on" segment content in `gate_on_rep` swings
  to +1 normalized, i.e. really is the MW-gate-on level -- confirm this
  actually happens during the FIRST `N_REPS` reps of each repeat, with
  the OFF level during the second `N_REPS` reps) -- should match the
  Sync signal's HIGH/LOW timing exactly.
- **CH1**: a continuous, undisturbed laser pulse train the whole time,
  same as always.
- **External trigger**: one visible pulse per second (SDG1062X output or
  AWG's Ext Trig input), same mechanism as prior tests.

## Stop / disconnect

In [ ]:
awg.write("OUTPUT1 OFF")
awg.write("OUTPUT2 OFF")
awg.close()
sdg.write("C1:OUTP OFF")
sdg.close()
print("AWG and SDG1062X outputs off, connections closed.")
